# Дерево решений с нуля: энтропия, разрезы и рекурсия

Построим собственный бинарный классификатор на NumPy: от выбора порога до предсказания по готовому дереву. `DecisionTreeClassifier` используем только для сравнения.

1. Энтропия и выбор разреза по одному признаку.
2. Выбор признака и рекурсивное построение дерева.
3. Вероятности, классы и проверки реализации.
4. Эксперименты с собственным деревом: XOR и контроль сложности.

Теоретические пояснения включены перед соответствующими заданиями. Основная практика рассчитана примерно на 50–60 минут: 15 минут на критерий и пороги, 20 на построение, 10 на предсказание и проверки, 10–15 на эксперименты. Дополнительные эксперименты с поворотом и изменением выборки можно выполнить после основной части.

Ячейки **ВАШ КОД** предназначены для заполнения. Каркас модели, данные, графики и проверки готовы. Данные генерируются локально; нужны NumPy, Matplotlib, scikit-learn и Jupyter. Задания последовательны: незавершённые функции выводят напоминание, реальные ошибки не скрываются.

## 0. Подготовка
Вспомогательные функции ниже отвечают за единообразные графики и таблицы. Цвет фона на графике — предсказанный класс, цвет точки — истинный класс, красное кольцо — ошибка на validation.

In [1]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.lines import Line2D
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, log_loss
from IPython.display import display, HTML

SEED = 42
BLUE, ORANGE, RED = '#2563eb', '#ea580c', '#dc2626'
plt.rcParams.update({'figure.figsize': (10, 4), 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.alpha': .15})

def table(rows, columns=None):
    # Небольшие таблицы без дополнительной зависимости от pandas.
    import html
    if not rows:
        return
    columns = list(rows[0]) if columns is None else columns
    def fmt(v):
        return f'{v:.4f}' if isinstance(v, (float, np.floating)) else str(v)
    header = ''.join(f'<th>{html.escape(str(c))}</th>' for c in columns)
    body = ''.join('<tr>'+''.join(f'<td>{html.escape(fmt(row[c]))}</td>' for c in columns)+'</tr>' for row in rows)
    display(HTML('<table><thead><tr>'+header+'</tr></thead><tbody>'+body+'</tbody></table>'))

def scatter_classes(ax, X, y, size=18):
    for label, color in [(0, BLUE), (1, ORANGE)]:
        mask = y == label
        ax.scatter(X[mask, 0], X[mask, 1], s=size, color=color, alpha=.65,
                   edgecolors='white', linewidths=.3)
    ax.set(xlabel='x₁', ylabel='x₂')

def decision_panels(models, X, y, bounds, ncols=3, angle=None, max_points=None):
    # Одинаковая сетка и цветовая семантика во всех панелях.
    X_show, y_show = (X, y) if max_points is None else (X[:max_points], y[:max_points])
    xmin, xmax, ymin, ymax = bounds
    xx, yy = np.meshgrid(np.linspace(xmin, xmax, 180), np.linspace(ymin, ymax, 180))
    grid = np.c_[xx.ravel(), yy.ravel()]
    ncols = min(ncols, len(models)); nrows = (len(models)+ncols-1)//ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.4*ncols, 4.2*nrows), squeeze=False)
    for ax, (name, model) in zip(axes.flat, models.items()):
        z = model.predict(grid).reshape(xx.shape)
        ax.contourf(xx, yy, z, levels=[-.5,.5,1.5], cmap=ListedColormap(['#dbeafe','#ffedd5']))
        if z.min() != z.max():
            ax.contour(xx, yy, z, levels=[.5], colors='#334155', linewidths=1)
        if angle is not None:
            a = np.deg2rad(angle)
            ax.contour(xx, yy, np.cos(a)*xx+np.sin(a)*yy, levels=[0], colors='#64748b', linestyles='--', linewidths=1.2)
        scatter_classes(ax, X_show, y_show, size=13)
        wrong = model.predict(X_show) != y_show
        ax.scatter(X_show[wrong,0], X_show[wrong,1], s=38, facecolors='none', edgecolors=RED, linewidths=.7)
        ax.set(title=f'{name}\naccuracy={accuracy_score(y, model.predict(X)):.3f} · листьев={model.get_n_leaves()}',
               xlim=(xmin,xmax), ylim=(ymin,ymax), aspect='equal')
        ax.grid(False)
    for ax in list(axes.flat)[len(models):]: ax.set_visible(False)
    handles = [Line2D([],[],marker='o',ls='',color=c,label=f'Класс {k}') for k,c in [(0,BLUE),(1,ORANGE)]]
    handles.append(Line2D([],[],marker='o',ls='',markerfacecolor='none',color=RED,label='Ошибка на validation'))
    if angle is not None:
        handles.append(Line2D([],[],color='#64748b',ls='--',label='Истинная граница'))
    fig.legend(handles=handles,loc='lower center',ncol=2 if angle is not None else 3,frameon=False)
    fig.tight_layout(rect=(0,.14 if nrows==1 else .07,1,1))
    plt.show()
print('Среда готова')

Среда готова


## Теория: как дерево принимает решение
Каждый внутренний узел задаёт вопрос $x_j\le t$. Объект идёт в левую или правую ветвь и повторяет этот процесс, пока не попадёт в лист. Пересечение осевых условий выделяет прямоугольные области, внутри которых ответ постоянен.

Пусть в листе 8 объектов класса 1 и 2 объекта класса 0. Его прогноз вероятности класса 1 — $8/10=0.8$, а предсказанный класс — 1. В общем случае:
$$p_k=\frac{n_k}{n},\qquad \hat y=\arg\max_k p_k.$$
При равенстве долей в нашей реализации выбираем класс 0. Вероятность 1 в чистом листе означает, что все его **обучающие** объекты одного класса, а не гарантированный ответ на новых данных.

Чтобы построить дерево, нужно выбрать полезный вопрос и решить, когда остановиться. Критерий полезности должен отличать смешанные узлы от чистых.

### Энтропия: сколько неопределённости осталось
Неожиданность события вероятности $p$ измерим как $-\log_2p$: гарантированное событие даёт 0 бит, событие вероятности $1/2$ — 1 бит. Средняя неожиданность метки — энтропия:
$$H(Y)=-\sum_kp_k\log_2p_k.$$
Нулевые частоты дают нулевой вклад. В узле используем эмпирические частоты классов.

| Доля класса 1 | Энтропия, бит | Интерпретация |
|---:|---:|---|
| 0 или 1 | 0 | Все метки одинаковы |
| 0.1 или 0.9 | 0.469 | Один класс преобладает |
| 0.5 | 1 | Максимальная неопределённость для двух классов |

Размер узла не определяет энтропию: 5/5 и 500/500 дают одинаковое значение, хотя частоты во втором случае оценены по большему числу объектов.

### Выигрыш: что мы узнали после вопроса
После разреза энтропию детей усредняем с весами их размеров:
$$H_{after}=\frac{n_L}{n}H(y_L)+\frac{n_R}{n}H(y_R),\qquad IG=H(y)-H_{after}.$$
Например, узел с 8 нулями и 2 единицами имеет энтропию около 0.722. Если слева 6 нулей, а справа 2 нуля и 2 единицы, то $H_{after}=0.6\cdot0+0.4\cdot1=0.4$, выигрыш — 0.322 бита. Усреднение детей поровну дало бы неверный результат.

Если обозначить результат вопроса через $B$, это $IG=H(Y)-H(Y\mid B)=I(Y;B)$: взаимная информация метки и **ответа на данный вопрос**. Выигрыш на train не гарантирует улучшения на новых данных.

Другая распространённая мера смешанности — Джини: $G=1-\sum_kp_k^2$. Для неё также максимизируют взвешенное уменьшение. Здесь реализуем только энтропию, чтобы сосредоточиться на устройстве дерева.

## 1. Найти лучший вопрос к данным
### 1.1. Реализуйте энтропию и информационный выигрыш
Для меток в одном узле оцениваем частоты классов $p_k=n_k/n$:
$$H(y)=-\sum_{k:p_k>0}p_k\log_2p_k.$$
Используем логарифм по основанию 2: энтропия измеряется в битах. Слагаемое при нулевой частоте считаем равным нулю.

Разрез $x_j\le t$ отправляет объект в левый узел, остальные — в правый. Выигрыш:
$$IG=H(y)-\frac{n_L}{n}H(y_L)-\frac{n_R}{n}H(y_R).$$

**Реализуйте:**
- `entropy(y)` — число; пустой массив по соглашению даёт 0. Метки — целые неотрицательные числа, возможны больше двух классов.
- `information_gain(y, left_mask)` — выигрыш; если одна часть пуста, верните 0. Маска — булев массив той же длины, что и `y`.

До кода предположите значения энтропии для `[0,0,0,0]`, `[0,0,1,1]` и `[0,0,0,1]`. Почему нельзя заменить веса дочерних узлов на $1/2$, если они имеют разный размер?

In [2]:
def entropy(y):
    # ВАШ КОД
    raise NotImplementedError('Реализуйте entropy')

def information_gain(y, left_mask):
    # ВАШ КОД
    raise NotImplementedError('Реализуйте information_gain')

In [3]:
def check_impurity():
    assert np.isclose(entropy([]), 0)
    assert np.isclose(entropy([0,0,0,0]), 0)
    assert np.isclose(entropy([0,0,1,1]), 1)
    assert np.isclose(entropy([0,0,0,1]), .8112781244591328)
    assert np.isclose(entropy([0,1,2]), np.log2(3))
    # Несимметричные размеры: неверное усреднение 1/2 должно провалить тест.
    y = np.array([0,0,0,1])
    assert np.isclose(information_gain(y, np.array([1,1,0,0],dtype=bool)), .3112781244591328)
    assert np.isclose(information_gain(y, np.array([1,0,0,0],dtype=bool)), .12255624891826566)
    assert np.isclose(information_gain(y, y==0), entropy(y))
    assert information_gain(y, np.ones(4,dtype=bool)) == 0
    print('Проверки энтропии и взвешенного выигрыша пройдены.')

impurity_ready = False
try:
    check_impurity()
    impurity_ready = True
except NotImplementedError as exc:
    print(exc)

Реализуйте entropy


### 1.2. Переберите допустимые пороги
Внутри интервала между соседними различными значениями признака разбиение не меняется. Поэтому достаточно проверить середины таких интервалов. Порог нельзя выбирать так, чтобы одинаковые значения признака попали в разные ветви.

Реализуйте `best_split_1d(x, y, min_leaf=1)`. Верните `(threshold, gain)` для максимального выигрыша. В каждом ребёнке должно быть не меньше `min_leaf` объектов. Если разрез невозможен, верните `(None, 0.0)`. При точном равенстве выигрышей выбирайте меньший порог. **Нулевой выигрыш сам по себе не запрещает разрез**: это пригодится в XOR.

Здесь достаточно прозрачного перебора с вызовом `information_gain`. Оптимизация скорости пока не требуется. Эту функцию используем внутри рекурсивного построения в следующем разделе.

In [4]:
def best_split_1d(x, y, min_leaf=1):
    # ВАШ КОД
    raise NotImplementedError('Реализуйте best_split_1d')

In [5]:
split_ready = False
try:
    t, gain = best_split_1d(np.array([0,0,1,1]), np.array([0,0,1,1]))
    assert np.isclose(t,.5) and np.isclose(gain,1)
    assert best_split_1d(np.ones(4), np.array([0,1,0,1])) == (None,0.)
    assert best_split_1d(np.arange(4), np.array([0,0,1,1]), min_leaf=3) == (None,0.)
    t, gain = best_split_1d(np.arange(4), np.zeros(4,dtype=int))
    assert t == .5 and np.isclose(gain,0), 'Нулевой выигрыш допустим; берём первый порог.'
    t, gain = best_split_1d(np.arange(4), np.array([0,1,1,1]), min_leaf=2)
    assert t == 1.5, 'Не забывайте ограничение min_leaf.'
    split_ready = True
    print('Проверки порогов, повторяющихся значений и min_leaf пройдены.')
except NotImplementedError as exc:
    print(exc)

Реализуйте best_split_1d


### 1.3. Посмотрите на критерий, а не только на результат
На одномерной выборке почти все маленькие значения относятся к классу 0, большие — к классу 1, но есть исключения. График покажет, как меняется выигрыш при движении порога. После выбора порога сравним его с деревом глубины 1 из scikit-learn.

**До запуска:** чистый маленький ребёнок всегда ли лучше большого, но слегка смешанного?

In [6]:
x_cut = np.arange(12, dtype=float)
y_cut = np.array([0,0,0,1,0,0,1,1,1,0,1,1])
if split_ready:
    thresholds = (x_cut[:-1]+x_cut[1:])/2
    gains = [information_gain(y_cut, x_cut <= t) for t in thresholds]
    threshold, gain = best_split_1d(x_cut, y_cut)
    stump = DecisionTreeClassifier(criterion='entropy', max_depth=1, random_state=SEED).fit(x_cut[:,None], y_cut)
    assert np.isclose(threshold, stump.tree_.threshold[0])
    fig, axes = plt.subplots(2,1,figsize=(10,5.5),sharex=True,gridspec_kw={'height_ratios':[1,2]})
    axes[0].scatter(x_cut, y_cut, c=[BLUE if y==0 else ORANGE for y in y_cut], s=85)
    axes[0].set(yticks=[0,1],ylabel='Метка',title='Один признак: какой вопрос уменьшает неопределённость?')
    axes[1].plot(thresholds,gains,'o-',color=BLUE)
    axes[1].scatter([threshold],[gain],s=110,color=RED,zorder=3,label=f'Лучший порог: {threshold:g}')
    for ax in axes: ax.axvline(threshold,color=RED,ls='--',alpha=.7)
    axes[1].set(xlabel='Порог t: вопрос x ≤ t',ylabel='Выигрыш, бит')
    axes[1].legend(); plt.tight_layout(); plt.show()
    left = x_cut <= threshold
    table([{'Узел': name,'Объектов':len(labels),'Доля класса 1':labels.mean(),'Энтропия':entropy(labels)}
           for name,labels in [('Родитель',y_cut),('Левый',y_cut[left]),('Правый',y_cut[~left])]])
else:
    print('Сначала завершите 1.1–1.2.')

Сначала завершите 1.1–1.2.


**Выводы:** почему лучший разрез не обязательно создаёт два чистых листа? Что измеряет высота точки на нижнем графике? Чем информационный выигрыш отличается от accuracy дерева? …

## 2. Построить дерево рекурсивно
### 2.1. Выберите лучший разрез среди всех признаков
Дерево жадно выбирает пару $(j,t)$ с максимальным текущим выигрышем. Затем независимо повторяет поиск в обоих детях. Оно не оценивает, какие следующие разрезы откроет выбранный вопрос.

Реализуйте `best_split(X, y, min_leaf)`: переберите столбцы `X`, вызовите `best_split_1d`, верните `(feature, threshold, gain)`. Если разрезов нет, верните `(None, None, 0.0)`. При одинаковом выигрыше выбирайте меньший номер признака; по одному признаку меньший порог уже выбирается в первой задаче. Допустимый разрез с нулевым выигрышем сохраняйте.

Область реализации дерева: двумерный массив конечных числовых признаков и метки **0/1**, без весов и пропусков. Энтропия из первого задания остаётся общей и для нескольких классов.

In [7]:
def best_split(X, y, min_leaf=1):
    # ВАШ КОД: перебор признаков с сохранением лучшего допустимого порога.
    raise NotImplementedError('Реализуйте best_split')

In [8]:
try:
    X_check=np.array([[7,0],[7,0],[7,1],[7,1]],dtype=float)
    y_check=np.array([0,0,1,1])
    j,t,g=best_split(X_check,y_check)
    assert j==1 and np.isclose(t,.5) and np.isclose(g,1)
    assert best_split(np.ones((4,2)),y_check)==(None,None,0.)
    j,t,g=best_split(np.c_[y_check,y_check],y_check)
    assert j==0, 'При равном выигрыше выбирайте первый признак.'
    print('Выбор признака и обработка постоянных столбцов проверены.')
except NotImplementedError as exc:
    print(exc)

Реализуйте best_split


### 2.2. Узел, остановка и рекурсивный шаг
Узел хранит частоты классов, число объектов и энтропию. Внутренний узел дополнительно хранит признак, порог и ссылки на детей. Если `feature is None`, это лист. Структура `Node` уже готова.

Рекурсия решает ту же задачу на меньшем наборе объектов:
1. Создать узел со статистиками текущего набора — он пока лист.
2. Если выполнено условие остановки, вернуть этот лист.
3. Найти лучший допустимый разрез. Если его нет или выигрыш слишком мал, вернуть лист.
4. Разделить объекты, рекурсивно построить детей, записать разрез и вернуть узел.

**Условия остановки:**
- метки в узле одинаковы;
- достигнута `max_depth`, если она задана; корень имеет глубину 0;
- объектов меньше `2 * min_samples_leaf`;
- нет допустимого порога (например, все признаки постоянны);
- локальный выигрыш меньше `min_gain` с допуском `1e-12`.

Параметр `min_gain` в нашей модели относится к **локальному** выигрышу. Он отличается от `min_impurity_decrease` в scikit-learn, где выигрыш умножают на долю объектов узла во всём train. При `min_gain=0` допускаем нулевой выигрыш. Рекурсия всё равно конечна: каждый допустимый разрез отправляет в обоих детей меньше объектов, чем было в родителе.

Реализуйте `build_node`: создание узла дано, условия остановки и разбиение нужно дописать. Параметры ограничений передавайте в оба рекурсивных вызова, а глубину увеличивайте на 1. На каждом шаге используйте **только объекты текущего узла**.

In [9]:
from dataclasses import dataclass

@dataclass
class Node:
    n_samples: int
    proba: np.ndarray
    impurity: float
    feature: object = None
    threshold: object = None
    left: object = None
    right: object = None

    @property
    def is_leaf(self):
        return self.feature is None

def build_node(X, y, depth=0, max_depth=None, min_samples_leaf=1, min_gain=0.0):
    counts = np.bincount(y, minlength=2)
    node = Node(n_samples=len(y), proba=counts/len(y), impurity=entropy(y))
    # ВАШ КОД: остановка, лучший разрез, рекурсивное построение детей.
    raise NotImplementedError('Реализуйте build_node')

In [10]:
def leaf_nodes(node):
    if node.is_leaf:
        return [node]
    return leaf_nodes(node.left)+leaf_nodes(node.right)

def node_depth(node):
    return 0 if node.is_leaf else 1+max(node_depth(node.left),node_depth(node.right))

def check_structure(node):
    assert np.isclose(node.proba.sum(),1) and np.all(node.proba>=0)
    if node.is_leaf:
        assert node.left is None and node.right is None
    else:
        assert node.left is not None and node.right is not None
        assert 0 < node.left.n_samples < node.n_samples
        assert 0 < node.right.n_samples < node.n_samples
        assert node.left.n_samples+node.right.n_samples==node.n_samples
        check_structure(node.left); check_structure(node.right)

structure_ready=False
try:
    X_check=np.arange(8,dtype=float)[:,None]
    y_check=np.array([0,1,0,1,0,1,0,1])
    stump=build_node(X_check,y_check,max_depth=0)
    assert stump.is_leaf and np.allclose(stump.proba,[.5,.5])
    pure=build_node(X_check,np.ones(8,dtype=int))
    assert pure.is_leaf and np.allclose(pure.proba,[0,1])
    constant=build_node(np.ones((8,2)),y_check)
    assert constant.is_leaf
    limited=build_node(X_check,y_check,max_depth=2,min_samples_leaf=2)
    check_structure(limited)
    assert node_depth(limited)<=2
    assert all(leaf.n_samples>=2 for leaf in leaf_nodes(limited))
    # Полное дерево должно суметь разделить восемь различных координат.
    full=build_node(X_check,y_check)
    check_structure(full)
    assert len(leaf_nodes(full))==8
    assert build_node(X_check,y_check,min_gain=2).is_leaf
    structure_ready=True
    print('Остановка, глубина, размер листа и целостность структуры проверены.')
except NotImplementedError as exc:
    print(exc)

Реализуйте entropy


## 3. Пройти от корня к предсказанию
### 3.1. Вероятности и метки
Обучение изменяет структуру; предсказание лишь проходит по уже выбранным условиям. В `predict_one(node, x)` спускайтесь влево при `x[feature] <= threshold`, вправо — иначе. Достигнув листа, верните его вектор вероятностей формы `(2,)`.

Это можно реализовать циклом или рекурсией. Не пересчитывайте частоты по новым данным и не изменяйте узлы при предсказании. Модель не должна знать истинные метки validation.

Класс `NumpyDecisionTree` ниже объединяет ваши функции в интерфейс `fit`, `predict_proba`, `predict`; его код готов. `predict` выбирает максимум вероятности, при равенстве — класс 0.

In [11]:
def predict_one(node, x):
    # ВАШ КОД: пройти по условиям до листа и вернуть две вероятности.
    raise NotImplementedError('Реализуйте predict_one')

In [12]:
class NumpyDecisionTree:
    def __init__(self, max_depth=None, min_samples_leaf=1, min_gain=0.0):
        self.max_depth=max_depth
        self.min_samples_leaf=min_samples_leaf
        self.min_gain=min_gain

    def fit(self, X, y):
        X=np.asarray(X,dtype=float); y=np.asarray(y)
        if X.ndim!=2 or X.shape[1]==0 or len(X)==0 or y.shape!=(len(X),):
            raise ValueError('Нужны непустой X формы (n,d) и y формы (n,)')
        if not np.isfinite(X).all() or not np.isin(y,[0,1]).all():
            raise ValueError('Поддерживаются конечные признаки и метки 0/1')
        if self.min_samples_leaf<1 or int(self.min_samples_leaf)!=self.min_samples_leaf:
            raise ValueError('min_samples_leaf — положительное целое')
        if self.max_depth is not None and (self.max_depth<0 or int(self.max_depth)!=self.max_depth):
            raise ValueError('max_depth — неотрицательное целое или None')
        if not np.isfinite(self.min_gain) or self.min_gain<0:
            raise ValueError('min_gain должен быть конечным и неотрицательным')
        self.n_features_in_=X.shape[1]
        self.classes_=np.array([0,1])
        self.root_=build_node(X,y.astype(int),0,self.max_depth,self.min_samples_leaf,self.min_gain)
        return self

    def predict_proba(self, X):
        X=np.asarray(X,dtype=float)
        if X.ndim!=2 or X.shape[1]!=self.n_features_in_ or not np.isfinite(X).all():
            raise ValueError('Некорректная форма или значения X')
        if len(X)==0:
            return np.empty((0,2))
        return np.vstack([predict_one(self.root_,x) for x in X])

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X),axis=1)]

    def get_n_leaves(self):
        return len(leaf_nodes(self.root_))

    def get_depth(self):
        return node_depth(self.root_)

    def apply(self, X):
        # Номер листа для графика распределения вероятностей.
        leaves=leaf_nodes(self.root_); indexes={id(leaf):i for i,leaf in enumerate(leaves)}
        result=[]
        for x in np.asarray(X):
            node=self.root_
            while not node.is_leaf:
                node=node.left if x[node.feature]<=node.threshold else node.right
            result.append(indexes[id(node)])
        return np.array(result)

def show_tree(node, prefix=''):
    if node.is_leaf:
        print(f'{prefix}Лист: n={node.n_samples}, p(1)={node.proba[1]:.3f}')
    else:
        print(f'{prefix}x{node.feature+1} ≤ {node.threshold:.3f}; n={node.n_samples}, H={node.impurity:.3f}')
        show_tree(node.left,prefix+'  ДА  ')
        show_tree(node.right,prefix+'  НЕТ ')

In [13]:
tree_ready=False
try:
    # Ручная структура проверяет направление ветвления, включая равенство порогу.
    manual=Node(4,np.array([.5,.5]),1.,feature=0,threshold=2.,
                left=Node(2,np.array([.75,.25]),.811),right=Node(2,np.array([.25,.75]),.811))
    assert np.allclose(predict_one(manual,np.array([2.])),[.75,.25])
    assert np.allclose(predict_one(manual,np.array([3.])),[.25,.75])
    X_easy=np.array([[0.],[1.],[2.],[3.]])
    y_easy=np.array([0,0,1,1])
    own=NumpyDecisionTree(max_depth=1).fit(X_easy,y_easy)
    reference=DecisionTreeClassifier(criterion='entropy',max_depth=1,random_state=SEED).fit(X_easy,y_easy)
    X_probe=np.array([[-1.],[.5],[1.5],[2.5],[4.]])
    assert np.allclose(own.predict_proba(X_probe),reference.predict_proba(X_probe))
    assert np.array_equal(own.predict(X_probe),reference.predict(X_probe))
    assert np.allclose(own.predict_proba(X_probe).sum(axis=1),1)
    assert np.all(own.predict_proba(X_probe)>=0)
    tie=NumpyDecisionTree(max_depth=0).fit(X_easy,y_easy)
    assert np.all(tie.predict(X_easy)==0)
    only_one=NumpyDecisionTree().fit(X_easy,np.ones(4,dtype=int))
    assert np.allclose(only_one.predict_proba(X_easy),np.tile([0,1],(4,1)))
    tree_ready=True
    print('Маршрутизация, вероятности, равенство порогу и сравнение с sklearn проверены.')
    show_tree(own.root_)
except NotImplementedError as exc:
    print(exc)

Реализуйте predict_one


**Что означает сравнение с библиотекой.** На простой выборке оптимальный разрез однозначен, и мы проверяем совпадение ответов. На более сложных данных одинаковый выигрыш может соответствовать разным вопросам: scikit-learn и наша реализация могут разрешать равенства по-разному. Полного совпадения всех деревьев требовать не нужно. Наша версия использует прозрачный перебор и не предназначена для больших наборов данных.

## 4. Исследовать собственное дерево
### 4.1. XOR: зачем разрешать нулевой выигрыш
Дерево глубины 2 может выразить XOR: сначала спросить знак одной координаты, затем другой. Но на четырёх равновероятных точках $(\pm1,\pm1)$ каждый из двух первых вопросов оставляет доли классов 50/50. Значит, выигрыш первого разреза равен нулю.

**Перед запуском:** что сделает ваша реализация при `min_gain=0`, а что при `min_gain=1e-8`? Почему правило `if gain <= 0: return leaf` лишило бы её возможности решить эту задачу?

Проверьте предположение готовым кодом. Здесь используются именно ваши функции, а не библиотечное дерево.

In [14]:
X_xor=np.repeat(np.array([[-1,-1],[-1,1],[1,-1],[1,1]],dtype=float),20,axis=0)
y_xor=(X_xor[:,0]*X_xor[:,1]<0).astype(int)
if tree_ready:
    print('Выигрыши корневых разрезов:', [information_gain(y_xor,X_xor[:,j]<=0) for j in range(2)])
    xor_models={}
    for min_gain in [0.,1e-8]:
        model=NumpyDecisionTree(max_depth=2,min_gain=min_gain).fit(X_xor,y_xor)
        xor_models[f'min_gain={min_gain:g}']=model
        print(f'\nmin_gain={min_gain:g}, accuracy={accuracy_score(y_xor,model.predict(X_xor)):.3f}')
        show_tree(model.root_)
    assert xor_models['min_gain=0'].get_n_leaves()==4
    assert np.array_equal(xor_models['min_gain=0'].predict(X_xor),y_xor)
    assert xor_models['min_gain=1e-08'].get_n_leaves()==1
    # Независимые точки нужны для визуализации областей между четырьмя train-точками.
    rng_xor=np.random.default_rng(15)
    X_xor_val=rng_xor.uniform(-1.4,1.4,size=(240,2))
    y_xor_val=(X_xor_val[:,0]*X_xor_val[:,1]<0).astype(int)
    decision_panels(xor_models,X_xor_val,y_xor_val,(-1.5,1.5,-1.5,1.5),ncols=2)
else:
    print('Сначала завершите реализацию дерева и проверки раздела 3.')

Сначала завершите реализацию дерева и проверки раздела 3.


### 4.2. Глубина, размер листа и вероятности
Ограничение глубины задаёт число вопросов на пути, `min_samples_leaf` запрещает создавать маленькие листья. Если позволять всё более мелкие области, можно подогнать дерево под шум.

Частоты в листьях минимизируют обучающую log loss среди постоянных прогнозов внутри каждого листа. Значение этого минимума — энтропия листа (с натуральным логарифмом). Поэтому чистые листья дают нулевую обучающую потерю, но могут выдавать чрезмерно уверенные ошибки на новых данных.

В эксперименте ниже сравниваются четыре **ваших** дерева. 400 объектов используются для обучения, 600 — для validation. Независимую итоговую test-оценку здесь не заявляем: это исследование поведения модели.

**Задание:** до запуска выберите ожидаемого победителя по accuracy и по log loss. Затем измените одно из ограничений и объясните, какие области границы изменились. Готовый код обучения не нужно переписывать.

In [15]:
X_moon,y_moon=make_moons(n_samples=1000,noise=.28,random_state=SEED)
X_train,X_val,y_train,y_val=train_test_split(X_moon,y_moon,train_size=400,stratify=y_moon,random_state=SEED)
moon_bounds=(X_moon[:,0].min()-.15,X_moon[:,0].max()+.15,X_moon[:,1].min()-.15,X_moon[:,1].max()+.15)
tree_models=None
if tree_ready:
    configurations=[(3,1),(3,15),(None,1),(None,15)]
    tree_models={(depth,leaf):NumpyDecisionTree(max_depth=depth,min_samples_leaf=leaf).fit(X_train,y_train)
                 for depth,leaf in configurations}
    rows=[]
    for (depth,leaf),model in tree_models.items():
        p=model.predict_proba(X_val)[:,1]
        wrong=model.predict(X_val)!=y_val
        rows.append({'Глубина':str(depth),'min_leaf':leaf,'Листьев':model.get_n_leaves(),
                     'Train accuracy':accuracy_score(y_train,model.predict(X_train)),
                     'Val accuracy':accuracy_score(y_val,model.predict(X_val)),
                     'Val log loss':log_loss(y_val,p,labels=[0,1]),
                     'Уверенных ошибок':int((wrong & ((p==0)|(p==1))).sum())})
    table(rows)
    decision_panels({f'Глубина {d}, min_leaf={l}':m for (d,l),m in tree_models.items()},X_val,y_val,moon_bounds,ncols=2)
else:
    print('Сначала завершите реализацию дерева.')

Сначала завершите реализацию дерева.


**Как читать результаты.** «Уверенная ошибка» — неправильный ответ с вероятностью класса 1 ровно 0 или 1. Теоретическая log loss для такого объекта бесконечна; scikit-learn численно обрезает вероятности около границ и показывает большое конечное значение.

- Какое ограничение помогло качеству меток, а какое — качеству вероятностей? …
- Почему `min_samples_leaf` не гарантирует калибровку и не запрещает чистые листья? …
- Что изменится в вашей реализации, если заменить критерий на Джини? Какие части останутся прежними? …

**Результат основной работы:** у вас есть классификатор с обучением, рекурсивной структурой, вероятностями и управляемой сложностью. Объясните на примере один критерий остановки и один случай, когда он мешает точному разделению.

## Дополнительные эксперименты
### Поворот координат
Этот раздел не входит в обязательную реализацию. Повернём те же объекты с теми же метками: информация и расстояния сохранятся, но наклонная граница потребует нескольких осевых разрезов. Предположите, как изменится число листьев при глубине 1 и 6.

In [16]:
def rotate(X,angle):
    a=np.deg2rad(angle)
    R=np.array([[np.cos(a),-np.sin(a)],[np.sin(a),np.cos(a)]])
    return X@R.T

if tree_ready:
    rng=np.random.default_rng(123)
    Xr_train=rng.uniform(-1,1,size=(250,2)); yr_train=(Xr_train[:,0]>0).astype(int)
    Xr_val=rng.uniform(-1,1,size=(1000,2)); yr_val=(Xr_val[:,0]>0).astype(int)
    for angle in [0,45]:
        models={f'{angle}°, глубина {depth}':NumpyDecisionTree(max_depth=depth,min_samples_leaf=5).fit(rotate(Xr_train,angle),yr_train)
                for depth in [1,6]}
        decision_panels(models,rotate(Xr_val,angle),yr_val,(-1.45,1.45,-1.45,1.45),ncols=2,angle=angle,max_points=300)
    print('На графиках 300 validation-точек; accuracy рассчитана по всем 1000.')

### Изменение обучающей выборки
Удалим случайные 10% train и сравним предсказания на том же validation. Выбор между близкими по выигрышу разрезами может измениться, а вместе с ним — вся ветвь. Здесь алгоритм детерминирован; меняются только данные.

Доля несовпадений не равна разнице accuracy: две модели могут делать одинаковое число ошибок на разных объектах. Усреднение прогнозов нескольких деревьев — возможный способ уменьшить эту чувствительность и мотивация для будущего бэггинга.

In [17]:
if tree_ready:
    base=NumpyDecisionTree().fit(X_train,y_train)
    rng=np.random.default_rng(314)
    rows=[]; reduced=None
    for trial in range(3):
        keep=rng.choice(len(y_train),size=int(.9*len(y_train)),replace=False)
        model=NumpyDecisionTree().fit(X_train[keep],y_train[keep])
        if trial==0: reduced=model
        rows.append({'Повтор':trial+1,'Доля несовпадений':np.mean(base.predict(X_val)!=model.predict(X_val)),
                     'Val accuracy':accuracy_score(y_val,model.predict(X_val))})
    table(rows)
    decision_panels({'Весь train':base,'Удалены 10%: первый повтор':reduced},X_val,y_val,moon_bounds,ncols=2)